In [ ]:
import rasterio
from rasterio.warp import transform_bounds
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
dem_file = "elevation/fairfax_dem_1m.tif"
with rasterio.open(dem_file) as src:
    downsample_factor = 10
    dem_data = src.read(
        1, 
        out_shape=(
            src.height // downsample_factor,
            src.width // downsample_factor
        ),
        resampling=rasterio.enums.Resampling.average
    )
    extent = [*src.bounds]
    extent = [extent[0], extent[2], extent[1], extent[3]]

    print(f"Original shape: {src.shape}")
    print(f"Downsampled shape: {dem_data.shape}")
    print(f"Memory reduction: {(1 - dem_data.nbytes / (src.height * src.width * 4)) * 100:.1f}%")
    dem_data = np.ma.masked_equal(dem_data, -9999)

In [ ]:
50509*42882

In [ ]:
county = gpd.read_file("GIS/COUNTY.geojson")
county.to_crs(src.crs, inplace=True)

watersheds = gpd.read_file("GIS/WATERSHEDS.geojson")
watersheds.to_crs(src.crs, inplace=True)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(dem_data, extent=extent, cmap='terrain', interpolation='bilinear')
county.boundary.plot(ax=ax, edgecolor='red', linewidth=2, label='County Boundary')
watersheds.boundary.plot(ax=ax, edgecolor='blue', linewidth=1, label='Watersheds')

cbar = plt.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label('Elevation (m)', rotation=270, labelpad=20)

ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.set_title('Fairfax County DEM with Boundary (EPSG:26918)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()